In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from umap import UMAP
from sklearn.manifold import TSNE
from config import DB_URL
engine = create_engine(DB_URL)

SIMILARITY_FEATURES = [
    "pts_per100_shrunk", "ast_per100_shrunk", "reb_per100_shrunk",
    "stl_per100_shrunk", "blk_per100_shrunk", "ts_pct", "efg_pct",
    "usg_pct", "three_rate", "free_throw_rate", "ast_pct",
    "oreb_pct", "dreb_pct", "trend_pts", "trend_pie", "pie", "net_rating",
]

query = f"""
    SELECT pf.player_id, pf.saison, pf.valuation_tier,
           pf.age_at_date, p.player_name, p.position,
           {', '.join(['pf.' + f for f in SIMILARITY_FEATURES])}
    FROM player_features pf
    JOIN players p ON p.player_id = pf.player_id
    WHERE pf.valuation_tier IN ('reliable', 'replacement_level')
"""

df = pd.read_sql(query, engine)
print(f"Pool : {len(df)} joueurs-saisons")

scaler = RobustScaler()
matrix = scaler.fit_transform(df[SIMILARITY_FEATURES].fillna(0))

# ------------------------------------------------------------------
# Figure 1 : Corrélation entre features
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 10))
corr = pd.DataFrame(matrix, columns=SIMILARITY_FEATURES).corr()
sns.heatmap(corr, annot=True, fmt=".1f", cmap="coolwarm",
            center=0, ax=ax, annot_kws={"size": 7})
ax.set_title("Corrélation entre features")
plt.tight_layout()
plt.savefig("viz_01_correlations.png", dpi=150)
plt.close()
print("→ viz_01_correlations.png")

# ------------------------------------------------------------------
# Figure 2 : PCA — variance expliquée
# ------------------------------------------------------------------

pca_full = PCA(n_components=min(17, len(df)))
pca_full.fit(matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Variance par composante
axes[0].bar(range(1, 18), pca_full.explained_variance_ratio_ * 100)
axes[0].set_xlabel("Composante")
axes[0].set_ylabel("Variance expliquée (%)")
axes[0].set_title("Variance par composante PCA")

# Variance cumulée
axes[1].plot(range(1, 18), np.cumsum(pca_full.explained_variance_ratio_) * 100, marker="o")
axes[1].axhline(80, color="red", linestyle="--", label="80%")
axes[1].axhline(90, color="orange", linestyle="--", label="90%")
axes[1].set_xlabel("Nombre de composantes")
axes[1].set_ylabel("Variance cumulée (%)")
axes[1].set_title("Variance cumulée PCA")
axes[1].legend()

plt.tight_layout()
plt.savefig("viz_02_pca_variance.png", dpi=150)
plt.close()
print("→ viz_02_pca_variance.png")

# ------------------------------------------------------------------
# Figure 3 : PCA 2D — colorié par position
# ------------------------------------------------------------------

pca2 = PCA(n_components=2)
coords_pca = pca2.fit_transform(matrix)

# Simplifier les positions
def simplify_position(pos):
    if pd.isna(pos):
        return "Unknown"
    pos = str(pos)
    if "Center" in pos:
        return "Center"
    if "Forward" in pos and "Guard" not in pos:
        return "Forward"
    if "Guard" in pos and "Forward" not in pos:
        return "Guard"
    return "Hybrid"

df["position_simple"] = df["position"].apply(simplify_position)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

colors_pos = {"Guard": "blue", "Forward": "green", "Center": "red", "Hybrid": "orange", "Unknown": "gray"}
for pos, color in colors_pos.items():
    mask = df["position_simple"] == pos
    axes[0].scatter(coords_pca[mask, 0], coords_pca[mask, 1],
                    c=color, label=pos, alpha=0.4, s=15)
axes[0].set_title("PCA 2D — par position")
axes[0].legend()

colors_tier = {"reliable": "blue", "replacement_level": "orange"}
for tier, color in colors_tier.items():
    mask = df["valuation_tier"] == tier
    axes[1].scatter(coords_pca[mask, 0], coords_pca[mask, 1],
                    c=color, label=tier, alpha=0.4, s=15)
axes[1].set_title("PCA 2D — par valuation_tier")
axes[1].legend()

plt.tight_layout()
plt.savefig("viz_03_pca2d.png", dpi=150)
plt.close()
print("→ viz_03_pca2d.png")

# ------------------------------------------------------------------
# Figure 4 : UMAP 2D — colorié par position et tier
# ------------------------------------------------------------------

print("Calcul UMAP (peut prendre 30s)...")
umap_model = UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                  metric="cosine", random_state=42)
coords_umap = umap_model.fit_transform(matrix)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for pos, color in colors_pos.items():
    mask = df["position_simple"] == pos
    axes[0].scatter(coords_umap[mask, 0], coords_umap[mask, 1],
                    c=color, label=pos, alpha=0.4, s=15)
axes[0].set_title("UMAP — par position")
axes[0].legend()

for tier, color in colors_tier.items():
    mask = df["valuation_tier"] == tier
    axes[1].scatter(coords_umap[mask, 0], coords_umap[mask, 1],
                    c=color, label=tier, alpha=0.4, s=15)
axes[1].set_title("UMAP — par valuation_tier")
axes[1].legend()

plt.tight_layout()
plt.savefig("viz_04_umap2d.png", dpi=150)
plt.close()
print("→ viz_04_umap2d.png")

# ------------------------------------------------------------------
# Figure 5 : UMAP colorié par features clés
# ------------------------------------------------------------------

key_features = ["pts_per100_shrunk", "ast_per100_shrunk",
                "reb_per100_shrunk", "three_rate", "usg_pct", "oreb_pct"]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    values = df[feat].values
    sc = axes[i].scatter(coords_umap[:, 0], coords_umap[:, 1],
                         c=values, cmap="RdYlGn", alpha=0.5, s=12)
    plt.colorbar(sc, ax=axes[i])
    axes[i].set_title(f"UMAP — {feat}")

plt.tight_layout()
plt.savefig("viz_05_umap_features.png", dpi=150)
plt.close()
print("→ viz_05_umap_features.png")

# ------------------------------------------------------------------
# Figure 6 : t-SNE 2D par position
# ------------------------------------------------------------------

print("Calcul t-SNE (peut prendre 1-2 min)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, metric="cosine")
coords_tsne = tsne.fit_transform(matrix)

fig, ax = plt.subplots(figsize=(10, 8))
for pos, color in colors_pos.items():
    mask = df["position_simple"] == pos
    ax.scatter(coords_tsne[mask, 0], coords_tsne[mask, 1],
               c=color, label=pos, alpha=0.4, s=15)

# Annoter quelques joueurs connus
known_players = [
    "LeBron James", "Stephen Curry", "Nikola Jokic",
    "Giannis Antetokounmpo", "Kevin Durant", "Joel Embiid",
    "Luka Doncic", "Jayson Tatum", "Anthony Davis"
]
for _, row in df[df["player_name"].isin(known_players)].iterrows():
    idx = df.index.get_loc(row.name)
    ax.annotate(row["player_name"],
                (coords_tsne[idx, 0], coords_tsne[idx, 1]),
                fontsize=7, alpha=0.9)

ax.set_title("t-SNE — par position (avec joueurs connus annotés)")
ax.legend()
plt.tight_layout()
plt.savefig("viz_06_tsne.png", dpi=150)
plt.close()
print("→ viz_06_tsne.png")

print("\n✓ Toutes les visualisations générées.")
print("Regarde dans l'ordre : 01 → 02 → 03 → 04 → 05 → 06")

KeyboardInterrupt: 

In [1]:
from sqlalchemy import create_engine
from feature_store import build_and_save_scalers
from role_engine import build_and_save_roles
from config import DB_URL
import pandas as pd
import numpy as np 

engine = create_engine(DB_URL)

# Étape 1 — scalers feature_store
print("=== Étape 1 : scalers ===")
build_and_save_scalers(engine)

# Étape 2 — rôles
print("\n=== Étape 2 : roles ===")
build_and_save_roles(engine)

=== Étape 1 : scalers ===
[feature_store] chargement de toutes les saisons...
[feature_store] 2867 lignes chargées
[feature_store] distribution des tiers :
valuation_tier
reliable             1596
replacement_level     681
low_sample            590
Name: count, dtype: int64
[feature_store] scaler entraîné : low_sample (590 lignes)
[feature_store] scaler entraîné : reliable (1596 lignes)
[feature_store] scaler entraîné : replacement_level (681 lignes)
[feature_store] ✓ scalers sauvegardés dans models\c5_scalers.pkl

=== Étape 2 : roles ===
[role_engine] chargement des features brutes...
[role_engine] 2277 joueurs-saisons chargés.
[role_engine] scaler sauvegardé.
[role_engine] calcul UMAP (peut prendre ~30s)...


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[role_engine] UMAP sauvegardé.
[role_engine] clustering HDBSCAN...
[role_engine] 15 clusters, 349 outliers.
[role_engine] ✓ 2277 assignments écrits en base.


{-1: 'versatile_hybrid',
 0: 'fringe_rotation',
 1: 'low_impact_shooter',
 2: 'raw_prospect',
 3: 'developmental_big',
 4: 'rim_protector',
 5: 'dominant_big',
 6: 'versatile_glue_guy',
 7: 'ascending_shooter',
 8: '3and_d_wing',
 9: 'franchise_star',
 10: 'star_declining',
 11: 'defensive_connector',
 12: '3and_d_specialist',
 13: 'declining_shooter',
 14: 'veteran_fading'}

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from config import DB_URL

engine = create_engine(DB_URL)

# Vérification directe contre les chiffres du test_hdbscan.py
query = """
    SELECT 
        pa.cluster_label,
        pa.archetype_name,
        COUNT(*) as n,
        ROUND(AVG(pf.pts_per100_shrunk)::numeric, 1) as pts,
        ROUND(AVG(pf.ast_per100_shrunk)::numeric, 1) as ast,
        ROUND(AVG(pf.oreb_pct + pf.dreb_pct)::numeric, 2) as reb,
        ROUND(AVG(pf.blk_per100_shrunk)::numeric, 1) as blk,
        ROUND(AVG(pf.three_rate)::numeric, 2) as three_rate,
        ROUND(AVG(pf.usg_pct)::numeric, 2) as usg,
        ROUND(AVG(pf.pie)::numeric, 3) as pie
    FROM player_archetypes pa
    JOIN player_features pf 
        ON pf.player_id = pa.player_id 
        AND pf.saison = pa.saison
    GROUP BY pa.cluster_label, pa.archetype_name
    ORDER BY pa.cluster_label;
"""

df = pd.read_sql(text(query), engine)
print(df.to_string(index=False))

In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from config import DB_URL

engine = create_engine(DB_URL)

# ------------------------------------------------------------------
# 1. Distribution des archétypes
# ------------------------------------------------------------------

query1 = """
    SELECT 
        archetype_name,
        COUNT(*) as n,
        ROUND(AVG(membership_prob)::numeric, 3) as avg_prob,
        ROUND(MIN(membership_prob)::numeric, 3) as min_prob,
        ROUND(MAX(membership_prob)::numeric, 3) as max_prob
    FROM player_archetypes
    GROUP BY archetype_name
    ORDER BY n DESC;
"""

df_dist = pd.read_sql(text(query1), engine)
print("=== Distribution des archétypes ===")
print(df_dist.to_string(index=False))

# ------------------------------------------------------------------
# 2. Vérification sur joueurs connus
# ------------------------------------------------------------------

query2 = """
    SELECT 
        p.player_name,
        pa.saison,
        pa.archetype_name,
        ROUND(pa.membership_prob::numeric, 3) as prob,
        pa.cluster_label
    FROM player_archetypes pa
    JOIN players p ON p.player_id = pa.player_id
    WHERE p.player_name IN (
        'Stephen Curry', 'LeBron James', 'Nikola Jokić',
        'Giannis Antetokounmpo', 'Rudy Gobert', 'Anthony Davis',
        'Klay Thompson', 'Draymond Green', 'Victor Wembanyama'
    )
    ORDER BY p.player_name, pa.saison;
"""

df_stars = pd.read_sql(text(query2), engine)
print("\n=== Vérification joueurs connus ===")
for player, group in df_stars.groupby("player_name"):
    print(f"\n{player}")
    print(group[["saison", "archetype_name", "prob"]].to_string(index=False))

# ------------------------------------------------------------------
# 3. Top 5 joueurs par archétype (membership_prob)
# ------------------------------------------------------------------

query3 = """
    SELECT 
        pa.archetype_name,
        p.player_name,
        pa.saison,
        ROUND(pa.membership_prob::numeric, 3) as prob
    FROM player_archetypes pa
    JOIN players p ON p.player_id = pa.player_id
    WHERE pa.cluster_label != -1
    ORDER BY pa.archetype_name, pa.membership_prob DESC;
"""

df_top = pd.read_sql(text(query3), engine)

print("\n=== Top 5 par archétype ===")
for archetype, group in df_top.groupby("archetype_name"):
    top5 = group.head(5)[["player_name", "saison", "prob"]]
    print(f"\n{'─'*50}")
    print(f"  {archetype}")
    print(top5.to_string(index=False))

# ------------------------------------------------------------------
# 4. Joueurs avec membership_prob faible (frontière entre clusters)
# ------------------------------------------------------------------

query4 = """
    SELECT 
        p.player_name,
        pa.saison,
        pa.archetype_name,
        ROUND(pa.membership_prob::numeric, 3) as prob
    FROM player_archetypes pa
    JOIN players p ON p.player_id = pa.player_id
    WHERE pa.membership_prob < 0.3
    AND pa.cluster_label != -1
    ORDER BY pa.membership_prob ASC
    LIMIT 20;
"""

df_frontier = pd.read_sql(text(query4), engine)
print("\n=== Joueurs à la frontière (prob < 0.3) ===")
print(df_frontier.to_string(index=False))

# ------------------------------------------------------------------
# 5. Sanity check — total
# ------------------------------------------------------------------

query5 = """
    SELECT 
        COUNT(*) as total,
        COUNT(CASE WHEN cluster_label = -1 THEN 1 END) as outliers,
        ROUND(COUNT(CASE WHEN cluster_label = -1 THEN 1 END) * 100.0 / COUNT(*), 1) as pct_outliers,
        COUNT(DISTINCT archetype_name) as n_archetypes
    FROM player_archetypes;
"""

df_sanity = pd.read_sql(text(query5), engine)
print("\n=== Sanity check ===")
print(df_sanity.to_string(index=False))

=== Distribution des archétypes ===
     archetype_name   n  avg_prob  min_prob  max_prob
   versatile_hybrid 349     0.000     0.000       0.0
      rim_protector 304     0.786     0.571       1.0
     franchise_star 297     0.963     0.748       1.0
 low_impact_shooter 234     0.942     0.506       1.0
        3and_d_wing 176     0.783     0.534       1.0
    fringe_rotation 141     0.784     0.254       1.0
  ascending_shooter 131     0.870     0.522       1.0
     veteran_fading 105     0.994     0.876       1.0
 versatile_glue_guy 102     0.891     0.406       1.0
defensive_connector  83     0.904     0.668       1.0
  declining_shooter  72     0.965     0.735       1.0
     star_declining  69     0.973     0.836       1.0
       dominant_big  62     0.994     0.948       1.0
  developmental_big  53     0.949     0.646       1.0
  3and_d_specialist  51     0.989     0.937       1.0
       raw_prospect  48     0.992     0.901       1.0

=== Vérification joueurs connus ===

Anthony 

In [3]:
# Colle ça dans ton notebook
query = """
    SELECT 
        p.player_name,
        pa.saison,
        pa.archetype_name,
        ROUND(pa.membership_prob::numeric, 3) as prob
    FROM player_archetypes pa
    JOIN players p ON p.player_id = pa.player_id
    WHERE p.player_name IN (
        'Stephen Curry', 'LeBron James', 'Nikola Jokić',
        'Giannis Antetokounmpo', 'Rudy Gobert', 'Anthony Davis',
        'Klay Thompson', 'Draymond Green', 'Victor Wembanyama',
        'Joel Embiid', 'Jayson Tatum', 'Luka Dončić'
    )
    ORDER BY p.player_name, pa.saison;
"""

df = pd.read_sql(text(query), engine)
for player, group in df.groupby("player_name"):
    print(f"\n{player}")
    print(group[["saison", "archetype_name", "prob"]].to_string(index=False))


Anthony Davis
 saison    archetype_name  prob
2021-22 developmental_big 0.883
2022-23 developmental_big 1.000
2023-24 developmental_big 0.883
2024-25 developmental_big 0.815

Draymond Green
 saison   archetype_name  prob
2021-22   franchise_star 0.978
2022-23   franchise_star 1.000
2023-24 versatile_hybrid 0.000
2024-25   franchise_star 1.000
2025-26   franchise_star 1.000

Giannis Antetokounmpo
 saison archetype_name  prob
2021-22 franchise_star   1.0
2022-23 franchise_star   1.0
2023-24 franchise_star   1.0
2024-25 franchise_star   1.0
2025-26 franchise_star   1.0

Jayson Tatum
 saison archetype_name  prob
2021-22 franchise_star   1.0
2022-23 franchise_star   1.0
2023-24 franchise_star   1.0
2024-25 franchise_star   1.0

Joel Embiid
 saison   archetype_name  prob
2021-22   franchise_star   1.0
2022-23   franchise_star   1.0
2023-24   franchise_star   1.0
2025-26 versatile_hybrid   0.0

Klay Thompson
 saison     archetype_name  prob
2021-22     franchise_star 1.000
2022-23     star_d

In [5]:
# Colle ça dans ton notebook
query = """
    SELECT 
        p.player_name,
        pa.saison,
        pa.archetype_name,
        ROUND(pa.membership_prob::numeric, 3) as prob
    FROM player_archetypes pa
    JOIN players p ON p.player_id = pa.player_id
    WHERE p.player_name IN (
        'Stephen Curry', 'LeBron James', 'Nikola Jokić',
        'Giannis Antetokounmpo', 'Rudy Gobert', 'Anthony Davis',
        'Klay Thompson', 'Draymond Green', 'Victor Wembanyama',
        'Joel Embiid', 'Jayson Tatum', 'Luka Dončić'
    )
    ORDER BY p.player_name, pa.saison;
"""

df = pd.read_sql(text(query), engine)
for player, group in df.groupby("player_name"):
    print(f"\n{player}")
    print(group[["saison", "archetype_name", "prob"]].to_string(index=False))


Anthony Davis
 saison    archetype_name  prob
2021-22 developmental_big 0.883
2022-23 developmental_big 1.000
2023-24 developmental_big 0.883
2024-25 developmental_big 0.815

Draymond Green
 saison   archetype_name  prob
2021-22   franchise_star 0.978
2022-23   franchise_star 1.000
2023-24 versatile_hybrid 0.000
2024-25   franchise_star 1.000
2025-26   franchise_star 1.000

Giannis Antetokounmpo
 saison archetype_name  prob
2021-22 franchise_star   1.0
2022-23 franchise_star   1.0
2023-24 franchise_star   1.0
2024-25 franchise_star   1.0
2025-26 franchise_star   1.0

Jayson Tatum
 saison archetype_name  prob
2021-22 franchise_star   1.0
2022-23 franchise_star   1.0
2023-24 franchise_star   1.0
2024-25 franchise_star   1.0

Joel Embiid
 saison   archetype_name  prob
2021-22   franchise_star   1.0
2022-23   franchise_star   1.0
2023-24   franchise_star   1.0
2025-26 versatile_hybrid   0.0

Klay Thompson
 saison     archetype_name  prob
2021-22     franchise_star 1.000
2022-23     star_d

In [6]:
# Colle dans le notebook — on compare les stats par archétype
# avec ce qu'on avait validé manuellement

query = """
    SELECT 
        pa.archetype_name,
        pa.cluster_label,
        COUNT(*) as n,
        ROUND(AVG(pf.pts_per100_shrunk)::numeric, 1) as pts,
        ROUND(AVG(pf.ast_per100_shrunk)::numeric, 1) as ast,
        ROUND(AVG(pf.oreb_pct + pf.dreb_pct)::numeric, 2) as reb,
        ROUND(AVG(pf.blk_per100_shrunk)::numeric, 1) as blk,
        ROUND(AVG(pf.three_rate)::numeric, 2) as three_rate,
        ROUND(AVG(pf.usg_pct)::numeric, 2) as usg,
        ROUND(AVG(pf.pie)::numeric, 3) as pie
    FROM player_archetypes pa
    JOIN player_features pf 
        ON pf.player_id = pa.player_id 
        AND pf.saison = pa.saison
    GROUP BY pa.archetype_name, pa.cluster_label
    ORDER BY pa.cluster_label;
"""

df = pd.read_sql(text(query), engine)
print(df.to_string(index=False))

     archetype_name  cluster_label   n  pts  ast  reb  blk  three_rate  usg   pie
   versatile_hybrid             -1 251 20.8  5.8 0.17  0.8        0.43 0.18 0.092
    fringe_rotation              0 134 20.4  4.8 0.15  1.0        0.42 0.15 0.042
 low_impact_shooter              1  95 18.2  4.3 0.15  0.9        0.47 0.16 0.085
       raw_prospect              2  86 21.5  4.6 0.21  1.0        0.31 0.15 0.112
  developmental_big              3 361 21.5  3.5 0.29  2.0        0.20 0.18 0.116
      rim_protector              4 230 21.0  4.9 0.14  0.9        0.51 0.16 0.025
       dominant_big              5  70 21.3  4.8 0.22  1.0        0.28 0.17 0.066
 versatile_glue_guy              6 235 18.2  3.2 0.14  0.6        0.57 0.16 0.072
  ascending_shooter              7 135 22.6  4.1 0.14  0.7        0.50 0.18 0.091
        3and_d_wing              8  95 21.5  4.9 0.18  0.9        0.43 0.17 0.114
     franchise_star              9 367 28.2  7.9 0.15  0.6        0.38 0.25 0.123
     star_declin

In [3]:
from sqlalchemy import create_engine
from validator import run_backtest, print_backtest_report
from config import DB_URL

engine = create_engine(DB_URL)

result = run_backtest(engine)
print_backtest_report(result)

[validator] chargement des features...
[validator] 2277 joueurs-saisons chargés.
[validator] saisons testées : ['2021-22', '2022-23', '2023-24', '2024-25']

[validator] saison N = 2021-22
[validator] 489 joueurs à tester

[validator] saison N = 2022-23
[validator] 437 joueurs à tester

[validator] saison N = 2023-24
[validator] 452 joueurs à tester

[validator] saison N = 2024-25
[validator] 436 joueurs à tester

[validator] N+1 ✅ MAE comparables=0.0201 vs baseline=0.0276 → +27.2%

[validator] N+2 ✅ MAE comparables=0.0224 vs baseline=0.0283 → +20.7%

[validator] N+3 ✅ MAE comparables=0.0218 vs baseline=0.0265 → +17.8%


BACKTEST VALIDATOR — PIE
k=5 comparables
Saisons testées : ['2021-22', '2022-23', '2023-24', '2024-25']

Horizon    N        MAE Comp     MAE Base     Amélio     Verdict
------------------------------------------------------------
N+1        1178     0.0201       0.0276       +27.2%     ✅ meilleur
N+2        794      0.0224       0.0283       +20.7%     ✅ meilleur
N+3  

In [5]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

from validator import run_backtest_by_archetype

df_arch = run_backtest_by_archetype(engine)

print("\n=== Résultats N+1 par archétype ===")
n1 = df_arch[df_arch["horizon"] == 1].sort_values("improvement", ascending=False)
print(n1[["archetype", "n", "mae_comp", "mae_base", "improvement"]].to_string(index=False))

print("\n=== Résultats N+2 par archétype ===")
n2 = df_arch[df_arch["horizon"] == 2].sort_values("improvement", ascending=False)
print(n2[["archetype", "n", "mae_comp", "mae_base", "improvement"]].to_string(index=False))

print("\n=== Résultats N+3 par archétype ===")
n3 = df_arch[df_arch["horizon"] == 3].sort_values("improvement", ascending=False)
print(n3[["archetype", "n", "mae_comp", "mae_base", "improvement"]].to_string(index=False))

[validator] backtest par archétype...

=== Résultats N+1 par archétype ===
          archetype   n  mae_comp  mae_base  improvement
  3and_d_specialist  28    0.0101    0.0308        67.07
     franchise_star 212    0.0169    0.0329        48.64
  declining_shooter  44    0.0142    0.0271        47.61
     star_declining  49    0.0162    0.0271        40.39
        3and_d_wing 115    0.0145    0.0242        39.86
       dominant_big  36    0.0156    0.0231        32.41
      rim_protector 180    0.0178    0.0263        32.35
defensive_connector  45    0.0149    0.0217        31.45
     veteran_fading  59    0.0168    0.0221        24.18
   versatile_hybrid 204    0.0191    0.0238        19.53
  ascending_shooter  71    0.0190    0.0212        10.39
    fringe_rotation  32    0.0374    0.0381         1.97
 versatile_glue_guy  29    0.0315    0.0320         1.61
       raw_prospect  16    0.0363    0.0328       -10.73
  developmental_big  12    0.0537    0.0403       -33.36
 low_impact_s

In [6]:
# list_players.py
from sqlalchemy import create_engine, text
from config import DB_URL

engine = create_engine(DB_URL)

with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT p.player_id, p.player_name, p.position, 
               COUNT(DISTINCT pf.saison) as nb_saisons,
               MAX(pf.saison) as derniere_saison
        FROM players p
        JOIN player_features pf ON p.player_id = pf.player_id
        GROUP BY p.player_id, p.player_name, p.position
        ORDER BY p.player_name
    """)).fetchall()

for r in rows:
    print(f"{r.player_id:8} | {r.player_name:<30} | {r.position:<3} | {r.nb_saisons} saisons | dernière: {r.derniere_saison}")

  203932 | Aaron Gordon                   | Forward | 5 saisons | dernière: 2025-26
 1630565 | Aaron Henry                    | Forward | 1 saisons | dernière: 2021-22
 1628988 | Aaron Holiday                  | Guard | 5 saisons | dernière: 2025-26
 1630174 | Aaron Nesmith                  | Guard-Forward | 5 saisons | dernière: 2025-26
 1630598 | Aaron Wiggins                  | Guard | 5 saisons | dernière: 2025-26
 1627846 | Abdel Nader                    | Forward | 1 saisons | dernière: 2021-22
 1642846 | Ace Bailey                     | Forward | 1 saisons | dernière: 2025-26
 1642380 | Adama Bal                      | Guard | 1 saisons | dernière: 2025-26
 1641766 | Adama Sanogo                   | Forward | 2 saisons | dernière: 2024-25
 1641745 | Adam Flagler                   | Guard | 2 saisons | dernière: 2024-25
 1641737 | Adem Bona                      | Forward | 2 saisons | dernière: 2025-26
 1630278 | Ade Murkey                     | Guard | 1 saisons | dernière: 2021

In [14]:
with engine.connect() as conn:
    rows = conn.execute(text("""
        SELECT player_id, player_name
        FROM players
        WHERE player_name = 'Draymond Green'
    """)).fetchall()

for r in rows:
    print(r)

(203110, 'Draymond Green')


In [15]:
# test_comparables.py
from sqlalchemy import create_engine
from similarity import find_similar_players, find_historical_comparables
from config import DB_URL

engine = create_engine(DB_URL)

TARGETS = [
    (203932,  "Aaron Gordon"),
    (1629130, "Duncan Robinson"),
    (203507,  "Giannis Antetokounmpo"),
    (203497,  "Rudy Gobert"),
    (1628368, "De'Aaron Fox"),
    (203110, 'Draymond Green'),
]

SEASON = "2025-26"

with open("comparables_test.txt", "w", encoding="utf-8") as f:
    for player_id, name in TARGETS:
        f.write(f"\n{'='*60}\n")
        f.write(f"  {name}\n")
        f.write(f"{'='*60}\n")

        # --- Similaires actuels ---
        f.write("\n▶ SIMILAIRES ACTUELS (2025-26)\n")
        results = find_similar_players(engine, player_id, SEASON, k=5)
        for r in results:
            f.write(f"  {r.player_name:<28} {r.season}  sim={r.similarity_pct:.1f}%  [{r.archetype_name}]\n")

        # --- Comparables historiques ---
        f.write("\n▶ COMPARABLES HISTORIQUES (même âge ±2 ans)\n")
        results = find_historical_comparables(engine, player_id, SEASON, age_window=2, k=5)
        for r in results:
            f.write(f"  {r.player_name:<28} {r.season}  age={r.age:.1f}  sim={r.similarity_pct:.1f}%  [{r.archetype_name}]\n")

print("✅ Résultats écrits dans comparables_test.txt")

✅ Résultats écrits dans comparables_test.txt
